# Where data preparation ends and modelling begins

Every notebook of this topic transforms data, and some of those transformations can be done once, for everyone who will use the table, while others must be learned again, from the training rows, for every model. Telling the two apart is what keeps a model's evaluation honest, and it is also what decides who does each step in a real project.

## Two pipelines

In a real project, data usually passes through two pipelines:

- **The data pipeline**, often called **ETL** (*extract, transform, load*): it extracts the data from where it lives (files, databases, web APIs), transforms it into a clean, documented table (types, names, joins, quality rules), and loads it where others can use it. Data engineers build it, and its table serves many uses: reports, dashboards, several models. Data quality and the ETL process are subjects of their own, beyond this topic.
- **The model's pipeline**: everything between that table and a prediction. In scikit-learn, a `Pipeline` whose steps are fitted on the training rows and then applied unchanged to every new row ([California housing with pipelines](../supervised-learning/basic-workflows/california_with_pipelines.ipynb)).

Many steps could go in either. One question decides.

## The criterion

**Does the step learn anything from the data, or depend on how the data will be split for a model?**

- If it doesn't, it can be part of the ETL: it gives the same result whoever runs it and for whatever purpose.
- If it does, it belongs to the model's pipeline, *after* the split. A step that learns from the data must learn from the training rows only, or information from the validation and test rows leaks into the model, and the evaluation stops measuring what it should ([Data leakage](../supervised-learning/ml-fundamentals/data_leakage.ipynb)).

| Step | Learns from the data? | Where it belongs | Examples |
| --- | --- | --- | --- |
| Extraction and format conversion | no | ETL | a spreadsheet or an API's answers into a table; dropping a column that duplicates another |
| Types, names, joining sources | no | ETL | dates parsed, a code read as text, descriptive column names, one table from several files |
| Data-quality rules: check, report, set aside | no (fixed rules) | ETL, and checked again before modelling | placeholders to missing values, impossible values, duplicates |
| **Enrichment:** reference data, features from each row alone | no | ETL | the region of an address from a fixed map; the distance to a fixed point; a ratio of two columns |
| Imputation, scaling, encoding | yes: medians, means, category lists | model's pipeline, fitted on the training rows | a median filling the gaps; one-hot encoding |
| **Learned features** | yes | model's pipeline, fitted on the training rows | bins from quantiles; clusters of locations; the average target of each category |
| Features from *other rows' targets* | yes, from the target itself | model's pipeline, with the most care: the highest risk of leaking | the average target of a category; the prices of the nearest neighbours |
| Transforming the target | a logarithm no, a standardisation yes | model's pipeline (`TransformedTargetRegressor`), so that predictions come back in the target's units | the logarithm of the target |

## Every step of this topic

The steps of the other notebooks, by the criterion:

| Step | Learns? | Where | Notebook |
| --- | --- | --- | --- |
| Dropping rows with no measurement at all | no | ETL | [Exploring a dataset](deep-dives/exploring_a_dataset.ipynb) |
| Turning placeholders and impossible values into missing values | no | ETL | [Missing values](missing_values.ipynb), [Outliers](deep-dives/outliers.ipynb) |
| A column saying whether a value was recorded (`cabin_known`) | no | either | [Missing values](missing_values.ipynb) |
| Filling gaps with a median, a most frequent value, neighbours or a model | yes | model | [Missing values](missing_values.ipynb) |
| Clipping at a bound known from the domain (more than 10 bathrooms counted as 10) | no | ETL | [Outliers](deep-dives/outliers.ipynb) |
| Winsorising at percentiles of the data | yes | model | [Outliers](deep-dives/outliers.ipynb) |
| Ordinal encoding with the order given by hand; titles grouped by meaning | no | either | [Encoding categories](categorical_encoding.ipynb), [Creating features](deep-dives/creating_features.ipynb) |
| One-hot encoding (it learns the list of categories); grouping rare categories; target encoding | yes | model | [Encoding categories](categorical_encoding.ipynb) |
| Any scaler | yes | model | [Scaling numerical features](feature_scaling.ipynb) |
| A logarithm | no | either | [Skewed distributions](deep-dives/skewed_distributions.ipynb) |
| A power or quantile transformation | yes | model | [Skewed distributions](deep-dives/skewed_distributions.ipynb) |
| Ratios, time elapsed, cyclic months, products | no | either | [Creating features](deep-dives/creating_features.ipynb) |
| Bins with edges from the data | yes | model | [Creating features](deep-dives/creating_features.ipynb) |

"Either" means the step may be done once in the ETL, or inside the pipeline so that the pipeline can take raw rows on its own; the last section, a deep dive, returns to that choice.

## A leak, on a column of pure noise

The census table of [Encoding categories](categorical_encoding.ipynb), with one column added: a code with 10,000 possible values, drawn at random for each person. It contains no information about income, by construction:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

adult = fetch_openml("adult", version=2, as_frame=True, data_home="data").frame
random_generator = np.random.default_rng(0)
X = adult.drop(columns="class").assign(code=random_generator.integers(0, 10_000, len(adult)).astype(str))
y = (adult["class"] == ">50K").astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train["code"].value_counts().describe()[["mean", "min", "max"]]

mean     3.990706
min      1.000000
max     14.000000
Name: count, dtype: float64

About four training rows per code. A logistic regression on the other features, as in [Encoding categories](categorical_encoding.ipynb), is the honest baseline:

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

numerical = X.select_dtypes("number").columns.to_list()
nominal = [column for column in X.select_dtypes("category").columns if column != "education"]


def logistic_with(*extra_columns):
    preprocessing = make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), nominal),
        (StandardScaler(), numerical),
        *extra_columns,
    )
    return make_pipeline(preprocessing, LogisticRegression(max_iter=3000))


scores = {"without the code": cross_val_score(logistic_with(), X_train, y_train, cv=5).mean()}
round(scores["without the code"], 3)

np.float64(0.851)

Now the code, target-encoded ([Encoding categories](categorical_encoding.ipynb)) in three ways. First by hand, with the share of high incomes among the training rows of each code, computed once on all the training rows, before the cross-validation:

In [ ]:
share_by_code = y_train.groupby(X_train["code"]).mean()
X_train_by_hand = X_train.assign(code_encoded=X_train["code"].map(share_by_code))
scores["encoded by hand, before the split"] = cross_val_score(
    logistic_with((StandardScaler(), ["code_encoded"])), X_train_by_hand, y_train, cv=5).mean()

Second with scikit-learn's `TargetEncoder`, but fitted and applied to all the training rows before the cross-validation:

In [ ]:
from sklearn.model_selection import StratifiedKFold

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)  # the cross-fitting folds, fixed so that results repeat
target_encoder = TargetEncoder(cv=folds).fit(X_train[["code"]], y_train)
X_train_fitted_first = X_train.assign(code_encoded=target_encoder.transform(X_train[["code"]]).ravel())
scores["TargetEncoder fitted before the split"] = cross_val_score(
    logistic_with((StandardScaler(), ["code_encoded"])), X_train_fitted_first, y_train, cv=5).mean()

Third, `TargetEncoder` inside the pipeline, so that each fold of the cross-validation fits it on its own training part:

In [ ]:
scores["TargetEncoder inside the pipeline"] = cross_val_score(
    logistic_with((TargetEncoder(cv=folds), ["code"])), X_train, y_train, cv=5).mean()
pd.Series(scores).round(3).rename("cross-validated accuracy")

without the code                         0.851
encoded by hand, before the split        0.892
TargetEncoder fitted before the split    0.893
TargetEncoder inside the pipeline        0.851
Name: cross-validated accuracy, dtype: float64

A column of pure noise has raised the accuracy by four points, twice: the cross-validation now says the model is better. Inside the pipeline, the noise adds nothing, as it should.

The reason: with about four people per code, the share of high incomes of a code is, to a large extent, the person's own income. The encoded column carries a piece of each row's answer, and the model learns to read it. Cross-validation can't see the problem, because its validation rows were encoded with shares that included their own targets too: the leak happened before the split, so the split can't catch it. Inside the pipeline, each fold encodes its validation rows with shares learned from other rows only, and on those the code is what it really is, noise.

New rows, the test set, are encoded like any future row: with the shares learned from the training rows, which never include the new row's own income:

In [ ]:
leaky_model = logistic_with((StandardScaler(), ["code_encoded"])).fit(X_train_by_hand, y_train)
X_test_encoded = X_test.assign(code_encoded=X_test["code"].map(share_by_code).fillna(y_train.mean()))
baseline_model = logistic_with().fit(X_train, y_train)
pd.Series({
    "the model with the leaked column, as cross-validation predicted": scores["encoded by hand, before the split"],
    "the same model, on the test rows": leaky_model.score(X_test_encoded, y_test),
    "the model without the code, on the test rows": baseline_model.score(X_test, y_test),
}).round(3).rename("accuracy")

the model with the leaked column, as cross-validation predicted    0.892
the same model, on the test rows                                   0.794
the model without the code, on the test rows                       0.852
Name: accuracy, dtype: float64

The leak promised 0.89 and delivered 0.79, worse than the model that never saw the code (0.85). The model had learned to trust a column that, on new rows, is random. This is the general shape of a leak: an evaluation better than the truth, discovered only when the model meets data it hasn't seen, or not at all. Target encoding makes it dramatic because it uses the target itself; imputation and scaling fitted on all the rows leak too, more quietly ([Data leakage](../supervised-learning/ml-fundamentals/data_leakage.ipynb) measures the same mistake with a feature selection, and explains why a scaler's goes unnoticed).

## Feature engineering sits on both sides *(deep dive)*

A feature defined by knowledge of the domain and computed from each row alone (a distance to a fixed point, the region of an address from a fixed map, a ratio) can be produced once, in the ETL, for every user of the table. A feature whose values depend on other rows (a cluster, an average, the nearest neighbours) is fitted, so it belongs in the model's pipeline, after the split. "Feature engineering" names both, and only the criterion separates them.

## Harmonising must not hide a difference of meaning *(deep dive)*

Joining sources is ETL, and so is computing the same measure for each of them. Merging the results under one name is another matter. If three sources each measure a distance to a different landmark (to a different main avenue in each city, say), the honest table keeps three columns, each empty outside its own source: a single "distance to the landmark" column would look like harmonisation and hide that the three numbers don't mean the same thing. Whether a model trained on one source can use another's landmark is a question about the model, not one for the ETL to decide silently. The same goes for names: an honest ETL gives every column a name that says what it is.

## The line is not always sharp *(deep dive)*

Steps that learn nothing *may* go in the ETL; they don't have to. A team often computes some of them inside the model's pipeline anyway, so that the pipeline can take a raw row in production and compute everything itself: a model deployed as a service receives the same raw fields its training data had, not a table prepared by someone else's ETL. The criterion still says which steps may move to the ETL, and which must never move out of the pipeline. What production asks of a pipeline (the same transformations on every new row, and only the inputs known at the moment of predicting) is explained in [Overfitting and the train/test split](../supervised-learning/ml-fundamentals/overfitting_and_the_split.ipynb#the-test-set-stands-for-production).

Two complete examples draw the line on real data: [Pima diabetes](../supervised-learning/basic-workflows/pima_diabetes.ipynb), where impossible zeros become missing values before the split and are filled inside the pipeline, and [Thyroid disease, data preparation](https://github.com/avidaldo/thyroid/blob/master/01_data_preparation.ipynb), which separates its ETL from the model's preprocessing explicitly.